In [ ]:
import os, sys, subprocess, json, time, re, threading, uuid, traceback, urllib.request

NTFY_URL = "https://ntfy.sh/burak_kaggle_ai_900192bf_live"
def notify(msg):
    print(msg, flush=True)
    try:
        req = urllib.request.Request(NTFY_URL, data=msg.encode('utf-8'), method='POST')
        urllib.request.urlopen(req, timeout=8)
    except Exception as e:
        print('Notify err:', e)

try:
    notify('STATUS|Adim 1/4: 32B Kodlama + Photoshop AI ortami hazirlaniyor...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchvision', 'torchaudio'], check=False)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--upgrade-strategy', 'only-if-needed',
        'bitsandbytes', 'peft', 'trl', 'accelerate', 'datasets', 'fastapi', 'uvicorn', 'pydantic'
    ], check=True)
    os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
    for _k in list(sys.modules.keys()):
        if _k.startswith(('torchvision', 'transformers.image', 'transformers.loss', 'transformers.modeling', 'transformers.models', 'peft', 'trl')):
            sys.modules.pop(_k, None)
    import transformers.utils.import_utils as _iu
    _iu._torchvision_available = False

    import gc, torch
    for _v in ('model', 'peft_model', 'trainer', 'optimizer'):
        if _v in globals():
            try: del globals()[_v]
            except Exception: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    from datasets import load_dataset
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # Kod + Photoshop + Katmanli Tasarim egitim veri seti
    STARTER_DATA = [
        {'messages': [{'role': 'system', 'content': 'Sen kod yazma ve Photoshop/gorsel tasarim konusunda uzman otonom bir yapay zeka ajanisin. Kullanicinin istegini tek seferde anlar ve dogrudan ilgili araci (tool) cagirirsin.'}, {'role': 'user', 'content': '[Görsel: uploads/kartvizit.jpeg]\nBu görseldeki her parçayı tek tek analiz edip metinleri ve logoyu değiştirilebilir parçalara böl.'}, {'role': 'assistant', 'content': '<tool_call>\n{"name": "decompose_image_to_design", "arguments": {"image_path": "uploads/kartvizit.jpeg", "output_html": "gorsel_parcali_editor.html", "open_browser": true}}\n</tool_call>'}]},
        {'messages': [{'role': 'system', 'content': 'Sen kod yazma ve Photoshop/gorsel tasarim konusunda uzman otonom bir yapay zeka ajanisin. Kullanicinin istegini tek seferde anlar ve dogrudan ilgili araci (tool) cagirirsin.'}, {'role': 'user', 'content': 'Kartvizitteki ismi Ahmet Kaya yap, telefonu 0532 444 55 66 olarak değiştir ve logoyu büyüt.'}, {'role': 'assistant', 'content': '<tool_call>\n{"name": "update_editable_design", "arguments": {"instructions": "ismi Ahmet Kaya yap, telefonu 0532 444 55 66 yap, logoyu büyüt", "updates": [{"target": "isim", "text": "Ahmet Kaya"}, {"target": "telefon", "text": "0532 444 55 66"}, {"target": "logo", "scale": 1.25}], "open_browser": true}}\n</tool_call>'}]},
        {'messages': [{'role': 'system', 'content': 'Sen kod yazma ve Photoshop/gorsel tasarim konusunda uzman otonom bir yapay zeka ajanisin. Kullanicinin istegini tek seferde anlar ve dogrudan ilgili araci (tool) cagirirsin.'}, {'role': 'user', 'content': 'Oluşturduğun kartviziti masaüstüne taşı.'}, {'role': 'assistant', 'content': '<tool_call>\n{"name": "move_or_copy_file", "arguments": {"source": "modern_kartvizit_editor.html", "destination": "desktop", "operation": "copy"}}\n</tool_call>'}]}
    ]
    DATASET_PATH = 'dataset_train.jsonl'
    with open(DATASET_PATH, 'w', encoding='utf-8') as f:
        for row in STARTER_DATA:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    gpu_cnt = torch.cuda.device_count() if torch.cuda.is_available() else 0
    total_vram_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(gpu_cnt)) / (1024**3) if gpu_cnt else 0
    # 2x T4 (29.1 GB VRAM) uzerinde calisabilen EN GUCLU model: Qwen2.5-Coder-32B-Instruct (4-bit)
    BASE_MODEL = 'unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit' if total_vram_gb >= 24 else 'unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit'
    notify(f'STATUS|Adim 2/4: GPU ({gpu_cnt}x, {total_vram_gb:.1f} GB VRAM) -> EN GUCLU MODEL ({BASE_MODEL}) yukleniyor...')

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    max_mem = {i: '14GiB' for i in range(gpu_cnt)} if gpu_cnt >= 2 else None
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map='balanced' if gpu_cnt >= 2 else 'auto',
        max_memory=max_mem,
        trust_remote_code=True,
    )
    model.is_parallelizable = True
    model.model_parallel = True

    notify('STATUS|Adim 3/4: 32B Model uzerinde Kodlama + Photoshop QLoRA ince ayari yapiliyor...')
    try:
        lora_config = LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        )
        peft_model = get_peft_model(model, lora_config)
        optimizer = torch.optim.AdamW(peft_model.parameters(), lr=1e-4)
        peft_model.train()
        for row in STARTER_DATA:
            txt = tokenizer.apply_chat_template(row['messages'], tokenize=False)
            enc = tokenizer(txt, return_tensors='pt', truncation=True, max_length=512).to('cuda:0')
            out = peft_model(**enc, labels=enc['input_ids'])
            out.loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        os.makedirs('./my_trained_ai/final_adapter', exist_ok=True)
        peft_model.save_pretrained('./my_trained_ai/final_adapter')
        model = peft_model
        notify('STATUS|Adim 3/4 Tamamlandi: 32B QLoRA Kodlama + Photoshop adaptoru basariyla egitildi!')
    except Exception as train_err:
        notify(f'STATUS|32B model hazir (adaptor notu: {train_err})')
    model.eval()

    notify('STATUS|Adim 4/4: 32B Tool-Calling API ve Cloudflare tuneli aciliyor...')
    if not os.path.exists('cloudflared'):
        subprocess.run(['wget', '-q', '-O', 'cloudflared', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
        subprocess.run(['chmod', '+x', 'cloudflared'], check=True)

    from fastapi import FastAPI, Header, HTTPException
    from fastapi.responses import StreamingResponse
    from pydantic import BaseModel
    from typing import Optional, Any
    import uvicorn

    app = FastAPI()
    MODEL_ID = 'qwen2.5-coder-32b-pro'
    GIZLI_ANAHTAR = 'burak-ozel-ai-2026'

    def _yetki_kontrol(authorization: str = None, x_api_key: str = None):
        token = (authorization or '').replace('Bearer ', '').strip() or (x_api_key or '').strip()
        if token != GIZLI_ANAHTAR:
            raise HTTPException(status_code=401, detail='Yetkisiz erisim')

    class ChatReq(BaseModel):
        model: str = MODEL_ID
        messages: list
        tools: Optional[list] = None
        tool_choice: Optional[Any] = None
        stream: bool = False
        temperature: float = 0.2
        max_tokens: int = 2048

    def _parse_qwen_tool_calls(text: str):
        calls = []
        pattern = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.DOTALL)
        for m in pattern.finditer(text or ''):
            try:
                obj = json.loads(m.group(1))
                fn_name = obj.get('name') or (obj.get('function') or {}).get('name')
                fn_args = obj.get('arguments') if 'arguments' in obj else (obj.get('function') or {}).get('arguments', {})
                if not isinstance(fn_args, str):
                    fn_args = json.dumps(fn_args, ensure_ascii=False)
                if fn_name:
                    calls.append({
                        'id': f'call_{uuid.uuid4().hex[:8]}',
                        'type': 'function',
                        'function': {'name': fn_name, 'arguments': fn_args}
                    })
            except Exception:
                pass
        cleaned = pattern.sub('', text or '').strip()
        return cleaned, calls

    @app.get('/v1/models')
    def list_models(authorization: str = Header(default=None), x_api_key: str = Header(default=None)):
        _yetki_kontrol(authorization, x_api_key)
        return {'object': 'list', 'data': [{'id': MODEL_ID, 'object': 'model'}]}

    @app.get('/v1/terms/status')
    def terms_status():
        return {'accepted': True}

    @app.post('/v1/chat/completions')
    def chat_completions(req: ChatReq, authorization: str = Header(default=None), x_api_key: str = Header(default=None)):
        _yetki_kontrol(authorization, x_api_key)
        try:
            if req.tools:
                prompt = tokenizer.apply_chat_template(req.messages, tools=req.tools, tokenize=False, add_generation_prompt=True)
            else:
                prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda:0')
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=min(req.max_tokens, 2048),
                temperature=max(0.05, float(req.temperature or 0.2)),
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen_ids = out[0][inputs['input_ids'].shape[1]:]
        raw_with_special = tokenizer.decode(gen_ids, skip_special_tokens=False)
        _, tool_calls = _parse_qwen_tool_calls(raw_with_special)
        cleaned_reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        cleaned_reply, extra_calls = _parse_qwen_tool_calls(cleaned_reply)
        if not tool_calls and extra_calls:
            tool_calls = extra_calls
        if tool_calls:
            cleaned_reply = re.sub(r'\{\s*"name"\s*:.*?\}\s*\}?', '', cleaned_reply, flags=re.DOTALL).strip()
        msg_obj = {'role': 'assistant', 'content': cleaned_reply}
        if tool_calls:
            msg_obj['tool_calls'] = tool_calls
        if not req.stream:
            return {
                'id': f'chatcmpl-{uuid.uuid4().hex[:8]}', 'object': 'chat.completion', 'model': MODEL_ID,
                'choices': [{'index': 0, 'message': msg_obj, 'finish_reason': 'tool_calls' if tool_calls else 'stop'}],
            }
        def event_stream():
            delta_obj = {'content': cleaned_reply}
            if tool_calls:
                delta_obj['tool_calls'] = tool_calls
            chunk = {
                'id': f'chatcmpl-{uuid.uuid4().hex[:8]}', 'object': 'chat.completion.chunk', 'model': MODEL_ID,
                'choices': [{'index': 0, 'delta': delta_obj, 'finish_reason': 'tool_calls' if tool_calls else 'stop'}],
            }
            yield f'data: {json.dumps(chunk, ensure_ascii=False)}\n\n'
            yield 'data: [DONE]\n\n'
        return StreamingResponse(event_stream(), media_type='text/event-stream')

    threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8002, log_level='warning'), daemon=True).start()
    time.sleep(2)

    proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8002'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            base_u = m.group(0) + '/v1'
            notify(f'READY32B|{base_u}|{GIZLI_ANAHTAR}|{MODEL_ID}')
            break
    for _ in range(960):
        time.sleep(30)
except Exception as e:
    notify('ERROR32B|' + traceback.format_exc()[-900:])
    raise
